# Gemma 7B Reasoning-Augmented Fine-Tuning (CoT) for Implicit Sentiment Analysis

This notebook performs **Reasoning-Augmented Fine-tuning** (Chain-of-Thought) for the Implicit Sentiment Analysis task on the **Laptop** and **Restaurant** datasets.

**Approach:**
Instead of only feeding `Sentence` + `Aspect` into the model, we use **GPT-4o** to generate a detailed contextual reasoning analysis following a structured 3-step guideline. This analysis serves as step-by-step reasoning to help **Gemma 7B** better understand implicit sentiment before making predictions.

**Pipeline:**
1. **Reasoning Generation:** Use GPT-4o to generate `Expert Analysis` for all Train and Test samples.
2. **Fine-tuning:** Train Gemma 7B with input: `Sentence + Aspect + Expert Analysis` → `Label`.
3. **Inference & Evaluation:** Evaluate on the test set (with reasoning).

**Configuration:**
- **Model:** Google Gemma 7B (BFloat16)
- **LoRA:** Rank 64, Alpha 128, Dropout 0.1
- **Max Seq Length:** 1536 (increased to accommodate the reasoning text)
- **Epochs:** 4

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# @title 1. Install Dependencies
!pip uninstall -y torch torchvision torchaudio tensorflow fastai
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q -U transformers peft bitsandbytes trl accelerate openai tqdm scikit-learn

In [ ]:
# @title 2. Imports & Configuration
import os
import torch
import xml.etree.ElementTree as ET
import pandas as pd
import json
import time
from datasets import Dataset
from tqdm import tqdm
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    logging,
)
from peft import LoraConfig, PeftModel
from trl import SFTTrainer, SFTConfig
from huggingface_hub import login
from openai import OpenAI, AzureOpenAI

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# @markdown **Enter your HF Token (requires Gemma access):**
HF_TOKEN = "" # @param {type:"string"}
login(token=HF_TOKEN)

# @markdown **OpenAI / Azure OpenAI configuration for reasoning generation:**
OPENAI_API_KEY = "" # @param {type:"string"}
OPENAI_ENDPOINT = "" # @param {type:"string"}
OPENAI_API_VERSION = "2024-02-15-preview" # @param {type:"string"}
OPENAI_DEPLOYMENT_NAME = "gpt-4o" # @param {type:"string"}

# Use OpenAI() for standard API, or AzureOpenAI() for Azure deployments
# client = OpenAI(api_key=OPENAI_API_KEY)
client = AzureOpenAI(
    api_key=OPENAI_API_KEY,
    api_version=OPENAI_API_VERSION,
    azure_endpoint=OPENAI_ENDPOINT
)

logging.set_verbosity_error()

In [ ]:
# @title 3. XML Parsing & Reasoning Prompts

def parse_semeval_xml(xml_path):
    """Parse SemEval-style XML into a DataFrame of (sentence, aspect, sentiment) tuples."""
    if not os.path.exists(xml_path):
        print(f"File not found: {xml_path}")
        return pd.DataFrame()

    tree = ET.parse(xml_path)
    root = tree.getroot()
    data = []

    for sentence in root.findall('sentence'):
        text = sentence.find('text').text
        aspect_terms = sentence.find('aspectTerms')

        if aspect_terms is not None:
            for aspect in aspect_terms.findall('aspectTerm'):
                term = aspect.get('term')
                polarity = aspect.get('polarity')
                implicit = aspect.get('implicit_sentiment')

                if polarity == 'conflict':
                    continue

                data.append({
                    'text': text,
                    'aspect': term,
                    'sentiment': polarity,
                    'is_implicit': implicit
                })

    return pd.DataFrame(data)

def get_gpt4o_reasoning(client, text, aspect):
    """Generate structured reasoning analysis for a (sentence, aspect) pair using GPT-4o."""
    system_prompt = (
        "You are an expert linguistic analyst specializing in Implicit Sentiment Analysis (ISA). "
        "Your goal is to articulate the logical path from surface text to underlying sentiment, "
        "capturing both explicit signals and subtle pragmatic cues."
    )

    user_prompt = f"""
    **Task:** Analyze the sentiment towards the aspect "{aspect}" in the provided sentence.

    **Sentence:** "{text}"

    **Analysis Guidelines:**

    STEP 1: Contextual Anchoring & Domain Standards
    -   Locate the specific aspect "{aspect}" and identify the domain context to frame the analysis.
    -   Establish baseline expectations by defining properties that typically constitute a positive or negative experience for this aspect.
    -   Determine the nature of the statement, distinguishing between objective facts, subjective opinions, and conditional hypotheticals.

    STEP 2: Evidence Extraction (Explicit & Implicit)
    -   Scan the text for **Explicit** signals, identifying direct opinion words or sentiment modifiers attached to the aspect.
    -   Perform **Implicit** inference if explicit words are absent by applying pragmatic reasoning to the factual descriptions. Evaluate whether the featured property enhances user utility or causes inconvenience (Benefit/Harm Logic).
    -   Assess if the described value, such as weight, price, time, or texture, deviates from the standard norm and determine the desirability of this deviation (Normative Deviation).
    -   Analyze the immediate dissatisfaction or satisfaction imposed on the user by the stated fact (Consequence Analysis).

    STEP 3: Synthesis & Knowledge Transfer
    -   Synthesize the contextual and evidential findings into a coherent argument.
    -   Resolve any potential conflicts between surface tone and underlying pragmatic meaning to ensure the true sentiment is captured.
    -   Clearly articulate the sentiment direction, and specifically mention whether the sentiment is expressed **Explicitly** (via emotional words) or **Implicitly** (via factual inference).

    **Output Constraints:**
    -   Provide the analysis as a single, cohesive, dense paragraph.
    -   Do NOT include the text "STEP 1", "STEP 2", "STEP 3" in the final output.
    -   Do NOT output the isolated label (Positive/Negative/Neutral) at the end; let the reasoning convey the polarity naturally.
    """

    try:
        response = client.chat.completions.create(
            model=OPENAI_DEPLOYMENT_NAME,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0.7,
            max_tokens=300
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"Error generating reasoning: {e}")
        return "Analysis unavailable."

def add_reasoning_column(df, client, save_path=None, force_update=False):
    """Add GPT-4o reasoning to each row, with periodic checkpoint saves."""
    if 'reasoning' not in df.columns:
        df['reasoning'] = None

    print(f"Generating reasoning for {len(df)} samples... (Force Update: {force_update})")

    for idx, row in tqdm(df.iterrows(), total=len(df)):
        should_run = force_update or pd.isna(row['reasoning']) or row['reasoning'] == "Analysis unavailable."

        if should_run:
            analysis = get_gpt4o_reasoning(client, row['text'], row['aspect'])
            df.at[idx, 'reasoning'] = analysis

            if idx % 20 == 0 and save_path:
                df.to_csv(save_path, index=False, encoding='utf-8-sig')

    if save_path:
        df.to_csv(save_path, index=False, encoding='utf-8-sig')

    return df

In [ ]:
# @title 4. Prepare Data with Reasoning (Run Once)

# Source XML paths
LAPTOP_TRAIN_XML = "/content/drive/MyDrive/ATAC2026/data/laptop/laptop_train_implicit-labeled.xml"
RESTAURANT_TRAIN_XML = "/content/drive/MyDrive/ATAC2026/data/restaurant/restaurant_train_implicit-labeled.xml"
LAPTOP_TEST_XML = "/content/drive/MyDrive/ATAC2026/data/laptop/laptop_test_implicit-labeled.xml"
RESTAURANT_TEST_XML = "/content/drive/MyDrive/ATAC2026/data/restaurant/restaurant_test_implicit-labeled.xml"

# Output directory for reasoning-augmented CSVs
PROCESSED_DATA_DIR = "/content/drive/MyDrive/ATAC2026/data/reasoning_augmented"
os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)

laptop_train_rn_path = os.path.join(PROCESSED_DATA_DIR, "laptop_train_reasoning.csv")
restaurant_train_rn_path = os.path.join(PROCESSED_DATA_DIR, "restaurant_train_reasoning.csv")
laptop_test_rn_path = os.path.join(PROCESSED_DATA_DIR, "laptop_test_reasoning.csv")
restaurant_test_rn_path = os.path.join(PROCESSED_DATA_DIR, "restaurant_test_reasoning.csv")

# Set True to regenerate all reasoning (e.g., after changing the prompt).
# Set False to resume from where it left off.
FORCE_UPDATE_REASONING = False

def process_and_save(xml_path, csv_output_path, force_update=False):
    """Load or generate reasoning-augmented data, with resume support."""
    if os.path.exists(csv_output_path) and not force_update:
        df = pd.read_csv(csv_output_path)
        if 'reasoning' in df.columns and df['reasoning'].isna().sum() == 0:
            print(f"Loaded {csv_output_path} — all samples already have reasoning.")
            return df
        print(f"Resuming generation for {df['reasoning'].isna().sum()} samples...")
    else:
        print(f"Parsing XML: {xml_path}")
        df = parse_semeval_xml(xml_path)

    df = add_reasoning_column(df, client, save_path=csv_output_path, force_update=force_update)
    return df

# Generate reasoning for all splits (this takes time and API cost)
df_laptop_train = process_and_save(LAPTOP_TRAIN_XML, laptop_train_rn_path, force_update=FORCE_UPDATE_REASONING)
df_restaurant_train = process_and_save(RESTAURANT_TRAIN_XML, restaurant_train_rn_path, force_update=FORCE_UPDATE_REASONING)
df_laptop_test = process_and_save(LAPTOP_TEST_XML, laptop_test_rn_path, force_update=FORCE_UPDATE_REASONING)
df_restaurant_test = process_and_save(RESTAURANT_TEST_XML, restaurant_test_rn_path, force_update=FORCE_UPDATE_REASONING)

In [ ]:
# @title 5. Hyperparameters & Prompt Formatting

MODEL_ID = "google/gemma-7b"
OUTPUT_DIR = "/content/drive/MyDrive/ATAC2026/fine_tuned_models"

# Hyperparameters
LORA_R = 64
LORA_ALPHA = 128
LORA_DROPOUT = 0.1
BATCH_SIZE = 8
GRAD_ACC_STEPS = 2
LEARNING_RATE = 2e-4
NUM_EPOCHS = 4
LR_SCHEDULER = "cosine"
MAX_SEQ_LENGTH = 1024

def format_reasoning_prompt(sample):
    """Format a training sample with reasoning into an instruction-tuning prompt."""
    instruction = (
        "Identify the sentiment polarity (positive, negative, or neutral) of the specific aspect "
        "in the following sentence, utilizing the provided expert analysis."
    )
    input_text = f"Sentence: {sample['text']}\nAspect: {sample['aspect']}\nExpert Analysis: {sample['reasoning']}"
    response = sample['sentiment']
    full_prompt = f"Instruction:\n{instruction}\n\nInput:\n{input_text}\n\nResponse:\n{response}<eos>"
    return {"text": full_prompt}

def format_reasoning_prompt_inference(sample):
    """Format an inference sample with reasoning (no label in response)."""
    instruction = (
        "Identify the sentiment polarity (positive, negative, or neutral) of the specific aspect "
        "in the following sentence, utilizing the provided expert analysis."
    )
    input_text = f"Sentence: {sample['text']}\nAspect: {sample['aspect']}\nExpert Analysis: {sample['reasoning']}"
    prompt = f"Instruction:\n{instruction}\n\nInput:\n{input_text}\n\nResponse:\n"
    return prompt

In [ ]:
# @title 6. Training Function
import shutil

def run_reasoning_fine_tuning(dataset_name, csv_path, output_subfolder):
    """Fine-tune Gemma with reasoning-augmented data from a pre-generated CSV."""
    print(f"\n{'='*20}\nStarting Reasoning Fine-tuning for: {dataset_name}\n{'='*20}")

    if not os.path.exists(csv_path):
        print(f"Error: Data not found at {csv_path}. Run the reasoning generation step first.")
        return None

    df = pd.read_csv(csv_path)
    df['reasoning'] = df['reasoning'].fillna("No analysis available.")

    dataset = Dataset.from_pandas(df)
    dataset = dataset.map(format_reasoning_prompt)
    dataset = dataset.select_columns(['text'])
    print(f"Training samples: {len(dataset)}")

    # Load model (BFloat16, no quantization)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,
        device_map={"": 0}
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    # LoRA configuration
    peft_config = LoraConfig(
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        r=LORA_R,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
    )

    # Training arguments (save checkpoints to local temp, then copy final to Drive)
    local_temp_dir = f"/content/temp_training_reasoning/{output_subfolder}"
    training_args = SFTConfig(
        output_dir=local_temp_dir,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACC_STEPS,
        gradient_checkpointing=True,
        optim="paged_adamw_8bit",
        logging_steps=10,
        save_strategy="epoch",
        save_total_limit=1,
        learning_rate=LEARNING_RATE,
        fp16=False,
        bf16=True,
        max_grad_norm=0.3,
        warmup_ratio=0.03,
        group_by_length=True,
        lr_scheduler_type=LR_SCHEDULER,
        max_length=MAX_SEQ_LENGTH,
        dataset_text_field="text",
        packing=False
    )

    # Train
    trainer = SFTTrainer(
        model=model,
        train_dataset=dataset,
        peft_config=peft_config,
        processing_class=tokenizer,
        args=training_args,
    )
    trainer.train()

    # Save final model to Drive
    final_drive_path = os.path.join(OUTPUT_DIR, output_subfolder)
    os.makedirs(final_drive_path, exist_ok=True)
    trainer.model.save_pretrained(final_drive_path)
    tokenizer.save_pretrained(final_drive_path)

    local_runs_dir = os.path.join(local_temp_dir, "runs")
    if os.path.exists(local_runs_dir):
        shutil.copytree(local_runs_dir, os.path.join(final_drive_path, "runs"), dirs_exist_ok=True)

    print(f"Model saved to: {final_drive_path}")

    # Cleanup
    del model, trainer
    torch.cuda.empty_cache()
    shutil.rmtree(local_temp_dir, ignore_errors=True)

    return final_drive_path

In [ ]:
# @title 7. Execute Training

# Laptop
if os.path.exists(laptop_train_rn_path):
    laptop_reasoning_model_path = run_reasoning_fine_tuning(
        "Laptop", laptop_train_rn_path, "gemma-7b-laptop-isa-lora-reasoning"
    )
else:
    print("Skipping Laptop: reasoning CSV not found.")

# Restaurant
if os.path.exists(restaurant_train_rn_path):
    restaurant_reasoning_model_path = run_reasoning_fine_tuning(
        "Restaurant", restaurant_train_rn_path, "gemma-7b-restaurant-isa-lora-reasoning"
    )
else:
    print("Skipping Restaurant: reasoning CSV not found.")

In [ ]:
# @title 8. Run Inference with Reasoning

INFERENCE_OUTPUT_DIR = "/content/drive/MyDrive/ATAC2026/inference_results"
os.makedirs(INFERENCE_OUTPUT_DIR, exist_ok=True)

def run_reasoning_inference(test_csv_path, model_path, base_model_id, dataset_name):
    """Run inference on reasoning-augmented test data and save predictions to CSV."""
    print(f"Loading test data from {test_csv_path}...")
    df_test = pd.read_csv(test_csv_path)
    df_test['reasoning'] = df_test['reasoning'].fillna("No analysis available.")
    print(f"Found {len(df_test)} samples.")

    print(f"Loading model from {model_path}...")
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_id,
        torch_dtype=torch.bfloat16,
        device_map={"": 0}
    )
    model = PeftModel.from_pretrained(base_model, model_path)
    tokenizer = AutoTokenizer.from_pretrained(base_model_id)
    tokenizer.padding_side = "left"

    # Build prompts
    inputs = [format_reasoning_prompt_inference(row) for _, row in df_test.iterrows()]

    batch_size = 16
    preds = []

    print("Running inference...")
    for i in tqdm(range(0, len(inputs), batch_size)):
        batch_prompts = inputs[i:i+batch_size]
        batch_encodings = tokenizer(batch_prompts, return_tensors="pt", padding=True, truncation=True, max_length=1536).to("cuda")

        with torch.no_grad():
            outputs = model.generate(
                **batch_encodings,
                max_new_tokens=10,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id
            )

        decoded_outputs = tokenizer.batch_decode(outputs, skip_special_tokens=True)

        for out in decoded_outputs:
            response = out.split("Response:\n")[-1].strip().lower()
            for sentiment in ['positive', 'negative', 'neutral']:
                if response.startswith(sentiment):
                    response = sentiment
                    break
            preds.append(response)

    df_test['prediction'] = preds

    output_csv_path = os.path.join(INFERENCE_OUTPUT_DIR, f"{dataset_name}_reasoning_inference_results.csv")
    df_test.to_csv(output_csv_path, index=False, encoding='utf-8-sig')
    print(f"Saved results to: {output_csv_path}")

    del model, base_model
    torch.cuda.empty_cache()
    return df_test

# Auto-resolve model paths if not set by training cells
if 'laptop_reasoning_model_path' not in globals():
    laptop_reasoning_model_path = os.path.join(OUTPUT_DIR, "gemma-7b-laptop-isa-lora-reasoning")
if 'restaurant_reasoning_model_path' not in globals():
    restaurant_reasoning_model_path = os.path.join(OUTPUT_DIR, "gemma-7b-restaurant-isa-lora-reasoning")

if os.path.exists(laptop_test_rn_path) and os.path.exists(laptop_reasoning_model_path):
    print(">>> Laptop Reasoning Inference...")
    df_laptop_res = run_reasoning_inference(laptop_test_rn_path, laptop_reasoning_model_path, MODEL_ID, "Laptop")

if os.path.exists(restaurant_test_rn_path) and os.path.exists(restaurant_reasoning_model_path):
    print("\n>>> Restaurant Reasoning Inference...")
    df_restaurant_res = run_reasoning_inference(restaurant_test_rn_path, restaurant_reasoning_model_path, MODEL_ID, "Restaurant")

In [ ]:
# @title 9. Evaluation Metrics
from sklearn.metrics import accuracy_score, f1_score

def evaluate_reasoning_results(df_results, dataset_name, output_file_path):
    """Compute Overall, ESA, and ISA metrics for reasoning model predictions."""
    if df_results is None:
        return

    df_results['is_implicit'] = df_results['is_implicit'].map(lambda x: str(x).lower() == 'true')
    y_true = df_results['sentiment'].str.lower()
    y_pred = df_results['prediction'].str.lower()

    valid_labels = ['positive', 'negative', 'neutral']
    mask = y_true.isin(valid_labels) & y_pred.isin(valid_labels)
    df_filtered = df_results[mask].copy()

    y_true = df_filtered['sentiment'].str.lower()
    y_pred = df_filtered['prediction'].str.lower()
    is_implicit = df_filtered['is_implicit']

    # Overall
    acc = accuracy_score(y_true, y_pred)
    f1_macro = f1_score(y_true, y_pred, average='macro')

    # Explicit Subset (ESA)
    mask_explicit = ~is_implicit
    acc_explicit = accuracy_score(y_true[mask_explicit], y_pred[mask_explicit]) if mask_explicit.sum() > 0 else 0
    f1_explicit = f1_score(y_true[mask_explicit], y_pred[mask_explicit], average='macro') if mask_explicit.sum() > 0 else 0

    # Implicit Subset (ISA)
    mask_implicit = is_implicit
    acc_implicit = accuracy_score(y_true[mask_implicit], y_pred[mask_implicit]) if mask_implicit.sum() > 0 else 0
    f1_implicit = f1_score(y_true[mask_implicit], y_pred[mask_implicit], average='macro') if mask_implicit.sum() > 0 else 0

    report = f"==== REASONING MODEL EVALUATION: {dataset_name} ====\n"
    report += f"Total Samples: {len(df_filtered)}\n"
    report += f"Implicit (ISA): {mask_implicit.sum()} | Explicit (ESA): {mask_explicit.sum()}\n\n"
    report += f"1. Overall:\n   - Accuracy: {acc:.4f}\n   - F1-Macro: {f1_macro:.4f}\n"
    report += f"2. Explicit Subset (ESA):\n   - Accuracy: {acc_explicit:.4f}\n   - F1-Macro: {f1_explicit:.4f}\n"
    report += f"3. Implicit Subset (ISA):\n   - Accuracy: {acc_implicit:.4f}\n   - F1-Macro: {f1_implicit:.4f}\n"
    report += "=" * 40 + "\n"

    print(report)
    with open(output_file_path, "w") as f:
        f.write(report)

EVAL_OUTPUT_DIR = "/content/drive/MyDrive/ATAC2026/evaluation_results"
os.makedirs(EVAL_OUTPUT_DIR, exist_ok=True)

if 'df_laptop_res' in globals():
    evaluate_reasoning_results(df_laptop_res, "Laptop", os.path.join(EVAL_OUTPUT_DIR, "laptop_metrics_reasoning.txt"))

if 'df_restaurant_res' in globals():
    evaluate_reasoning_results(df_restaurant_res, "Restaurant", os.path.join(EVAL_OUTPUT_DIR, "restaurant_metrics_reasoning.txt"))